# Run this cell first

In [ ]:
# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

try:
  from testsrc import test_main
except (ModuleNotFoundError, ImportError):
  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=MTH2100/numerical_analysis/chaos"
  from testsrc import test_main

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except (AssertionError, ImportError):
      pass

# 7. Chaotic Systems: Sensitivity, Bifurcations, and Strange Attractors

## Why this matters

Chaos is an important but commonly misunderstood concept. Often people confuse
**chaos** with **randomness**. However, as we'll see, chaotic motion is
**deterministic**, meaning if we run a simulation twice under _precisely_ the
same conditions we will get the same answer. However, the key word there is
_precisely_.  Chaotic systems are not random, yet long-term prediction is
impossible because tiny differences in initial conditions grow exponentially. 

Understanding chaos is essential for:
- Climate and weather modelling (the famous "butterfly effect")
- Population dynamics near tipping points
- Engineering systems that must avoid instability
- Distinguishing "noise" from deterministic chaos in data

This notebook explores chaos through two classic examples: the **logistic map** (a
discrete-time model) and the **double pendulum** (a continuous-time mechanical system).

**Learning objectives**
- Understand what it means for a system to be chaotic
- Compute and interpret bifurcation diagrams
- Quantify sensitivity to initial conditions using the Lyapunov exponent
- Simulate the double pendulum and observe chaotic motion

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;">

**📹 VIDEO: Chaos: Sensitivity, Bifurcations, and Strange Attractors**

*What this video covers:* What is deterministic chaos; the logistic map as a simple example; fixed points and period-doubling; the bifurcation diagram; Lyapunov exponents; the double pendulum; sensitivity to initial conditions; the Lorenz attractor.

*(Embed the video here using an `<iframe>` or `IPython.display.Video`)*
</div>

## 1. The Logistic Map

The **logistic map** is the simplest example of deterministic chaos. It is encoded in the recursion relation,

$$x_{n+1} = r\, x_n (1 - x_n), \qquad x_n \in [0,1], \quad r \in [0,4].$$

Given a starting value $x_0$, and a parameter $r$ (often called the growth rate), we can compute $x_1$, from which we can compute $x_2$ etc. 

Clearly, the logistic map is not **random**. However, it is the prototypical example of chaos, displaying stable, periodic and chaotic motion for different values of $r$. Specifically it exhibits
- Stable fixed points for small $r$
- Period-2, 4, 8, ... cycles as $r$ increases (period doubling)
- Chaos for $r \gtrsim 3.57$

Despite its simplicity ($x_{n+1}$ is a quadratic function of $x_n$), the logistic map
generates arbitrarily complex behaviour.

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

1. Implement a function `logistic_map`. It should take `x` (think of this as $x_n$ in the formula) and `r` as inputs, and return an _updated_ value of x (i.e. $x_{n+1}$) using the formula 
$$x_{n+1} = r\, x_n (1 - x_n).$$
2. Implement a second function `iterate_map` which takes inputs `x0`, `r` and `n_iter`, and which calls `logistic_map` `n_iter` times, beginning with the starting value `x0`, and storing the updated value of `x` returned by each call in a list. (Don't forget that `x0` should be in the list too!) It should then return that list. 

If you do that correctly, you should see plots showing the behaviour of the logistic map for different values of the parameter $r$. 

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# your code goes here





# ── Orbit diagram: show what long-term behaviour looks like ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

r_examples = [2.5, 3.2, 3.5, 3.56, 3.8, 3.999]
descriptions = [
    'r=2.5 : stable fixed point',
    'r=3.2 : period-2 cycle',
    'r=3.5 : period-4 cycle',
    'r=3.56: period-8 (near chaos)',
    'r=3.8 : chaos (some order)',
    'r=3.99: fully chaotic'
]
for ax, r_val, desc in zip(axes.flat, r_examples, descriptions):
    xs = iterate_map(x0=0.5, r=r_val, n_iter=100)
    n  = len(xs)
    ax.plot(range(n), xs, 'b-o', ms=3, lw=1.0, alpha=0.8)
    ax.set_title(desc, fontsize=10)
    ax.set_xlabel('Iteration $n$')
    ax.set_ylabel('$x_n$')
    ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3)

plt.suptitle("Logistic map orbits for different values of $r$", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
runtest(['test_logistic', 'test_iterate'])

The most common way the chaos of the logistic map is demonstrated is with a bifurcation diagram. For a given value of $r$, the bifurcation diagram plots the values generated by the logistic map. For $r<3$, we have stable behaviour, and a single value is plotted for each value of $r$. For $3<r\lesssim3.57$ we have periodic beviour- there are several values that the logistic map generates in turn, and we see the line plotted splitting into two, then four, then eight as we approach the chaotic limit. This known as period doubling, for obvious reasons. Finally, when we move above the chaotic limit, the diagram 'fills in' because the logistic map is generating essentially every value between zero and one. There are further weird things that happen: specific values of $r$ for which we seemingly recover some order in the behaviour. 

In [ ]:
r_values = np.linspace(2.5, 4.0, 2000)
n_transient = 1000   # discard burn-in
n_display   = 200    # plot these iterations per r value

r_plot  = []
x_attractor = []

x = 0.5  # initial condition (arbitrary)
for r in r_values:
    # Reset with same x0
    x = 0.5
    for _ in range(n_transient):
        x = logistic_map(x, r)
    for _ in range(n_display):
        x = logistic_map(x, r)
        r_plot.append(r)
        x_attractor.append(x)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(r_plot, x_attractor, ',b', alpha=0.08, ms=0.3)
ax.set_xlabel('Parameter $r$', fontsize=14)
ax.set_ylabel('Long-term values of $x_n$', fontsize=14)
ax.set_title('Bifurcation diagram of the logistic map', fontsize=14)
ax.set_xlim(2.5, 4.0); ax.set_ylim(0, 1)
ax.grid(alpha=0.2)

# Annotate key bifurcation points
for r_bif, label in [(3.0, 'r≈3'), (3.449, 'r≈3.45'),
                      (3.544, ''), (3.5688, 'r≈3.57')]:
    ax.axvline(r_bif, color='red', ls='--', lw=0.8, alpha=0.5)
    ax.text(r_bif + 0.01, 0.05, label, fontsize=8, color='red', va='bottom')

plt.tight_layout()
plt.show()

## 2. Sensitive Dependence on Initial Conditions

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise
The key feature of chaotic systems is that a slight change in the initial conditions leads to dramatic changes in the behaviour. Demonstrate that for the logistic map using the `iterate_map` function that you defined earlier.

1. Use a starting value of `x0=0.5` and a reproduction parameter of `r=3.9`. Call `iterate_map` to generate 100 values of the logistic map. Store that in the array `logistic_a`
2. Then use a starting value of `x0=0.5000001`, and repeat- store the result in `logistic_b`
3. Plot the two arrays to observe the behaviour
4. Can you describe what you observe?


<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

The behaviour of the logistic map with these two starting values, separated by only $10^{-7}$, is the basically the same for the first 50 iterations. But from that point onwards, the two maps behave quite differently. And it is chaotic, in the sense that there are places where the two values are quite close, but places where they are wildly different. This unpredictability is the hallmark of chaos.  

</div>
</details>

</div>


In [ ]:
# your code goes here:







# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()

In [ ]:
runtest(['test_plot'])

When we say that chaotic systems are **sensitivite to initial conditions**, we mean two trajectories that start arbitrarily close
will diverge exponentially fast. This is the defining property of chaos, and it is why
[long-term weather prediction is fundamentally impossible](https://youtu.be/fDek6cYijxI?si=YA_FK7RqG9dT2fcK).

To quantify this chaos, we use the **Lyapunov exponent**, which compares the change in the computed value $\Delta x_n$ with the change in the initial value $\Delta x_0$. Then, $\lambda$ quantifies this rate of divergence by:
$$|\Delta x_n| \approx |\Delta x_0| \cdot e^{\lambda n}.$$

Thus we have three different regimes:
- $\lambda > 0$: chaotic (exponential divergence)
- $\lambda < 0$: stable (trajectories converge)
- $\lambda = 0$: marginal (e.g. periodic orbit)

The codebox below uses the two arrays you computed in the last exercise, and then fits a straight-line to the logarithm of $\Delta x_n$. The gradient of that straight line is the Lyapunov exponent. 

In [ ]:
Delta_xn    = np.abs(np.array(logistic_a) - np.array(logistic_b))

plt.semilogy(np.arange(len(Delta_xn)), Delta_xn, 'k-', lw=2, label='$|x_n^{(a)} - x_n^{(b)}|$')


n_fit = np.where(Delta_xn > 0.1)[0][0]

n_range = np.arange(1,n_fit)
log_D = np.log(Delta_xn[1:n_fit])
lyap = np.polyfit(n_range, log_D, 1)[0]

plt.semilogy(n_range, np.exp(lyap * n_range + log_D[0]), 'r--', lw=1.5,
            label=f'Slope $\\lambda \\approx {lyap:.3f}$')

plt.xlabel('Iteration $n$')
plt.ylabel('$|\\Delta x_n|$')

plt.legend()
print(f"Lyapunov exponent: {lyap}")


<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

**For each of the lines of code below, see if you can explain what is being done**

`Delta_xn    = np.abs(np.array(logistic_a) - np.array(logistic_b))`
<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

`np.abs` takes the modulus, `np.array` turns the lists of values `logistic_a` and `logistic_b` into numpy arrays, so that taking the difference between them gives us the element-wise difference of their values.

</div>
</details>

```python
n_fit = np.where(Delta_xn > 0.1)[0][0]

```

Hint: look at how `n_fit` is used in the codebox above.

<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

Exponential growth looks like a straight line when you plot it on a log scale, so we are only interested in the bit of the plot that looks like a straight line. If $\Delta x_n$ gets too large, then the two arrays we are comparing, `logistic_a` and `logistic_b` are no longer correlated. So, we need to work out which bit of our array $\Delta x_n$ contains something that looks like exponential growth. 

The function `np.where` gives back an array containing the array indices of all the places in `Delta_xn` where the value is larger than 0.1. More accurately, it gives back a 2D array: the first dimension contains the array indices, and the second dimension contains the values of `Delta_xn` at those locations. We're only interested in the locations, which we get with `np.where(...)[0]`.

We then select the first element of that array (with the final `[0]`) which is the location in the array where $\Delta x_n$ is larger than 0.1 _for the first time_. 

</div>
</details>

```python 
lyap = np.polyfit(n_range, log_sep, 1)[0]
```

Hint: check out the documentation for [`np.polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html)
<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

The `polyfit` function gives a polynomial fit to some data. The three arguments we provide to polyfit define the $x$ values, the $y$ values and then the order of that polynomial fit, in this case, 1. It's important that we provide the $\log$ of the $\Delta x_n$ values, because then the relationship is linear. If we provided `Delta_xn` directly, then polyfit wouldn't be able to fit the data. 

`polyfit` returns an array of values with the polynomial coefficients, with the highest power first. In our case, we are looking at a linear relationship  $$ \log(\Delta x_n) \propto \lambda n $$
so the leading coefficient is $\lambda$: the Lyapunov exponent

</div>
</details>

</div>

## 3. The Double Pendulum

The **double pendulum** is a classic example of a chaotic mechanical system. It consists
of two rigid rods of lengths $l_1, l_2$ with masses $m_1, m_2$ at their ends, hinged at
the top and at the junction.

![double_pendulum_animation](https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/MTH2100/numerical_analysis/chaos/double_pendulum.gif)

We describe the configuration by the angles $\theta_1, \theta_2$ measured from the
downward vertical, and define $\omega_i = \dot\theta_i$.
Writing $\Delta = \theta_2 - \theta_1$ and
$$D = (m_1 + m_2) - m_2\cos^2\!\Delta,$$
the full nonlinear equations of motion (no small-angle approximation) are:


$$\dot\theta_1 = \omega_1, \qquad \dot\theta_2 = \omega_2,$$

$$\dot\omega_1 = \frac{%
  m_2 l_1\omega_1^2\sin\Delta\cos\Delta
  + m_2 g\sin\theta_2\cos\Delta
  + m_2 l_2\omega_2^2\sin\Delta
  - (m_1+m_2)g\sin\theta_1
}{l_1 D},$$

$$\dot\omega_2 = \frac{%
  -m_2 l_2\omega_2^2\sin\Delta\cos\Delta
  + (m_1+m_2)g\sin\theta_1\cos\Delta
  - (m_1+m_2)l_1\omega_1^2\sin\Delta
  - (m_1+m_2)g\sin\theta_2
}{l_2 D}.$$

These four first-order ODEs form the state vector
$[\theta_1,\,\omega_1,\,\theta_2,\,\omega_2]$ that is passed to the solver.

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

Complete the `double_pendulum` function below, using the equations provided above. When that is done, the following cells will show the chaotic by the same method we used for the logistic equation: changing the initial conditions by some tiny amount, then verifying that the computed solution diverges.
</div>

In [ ]:
def double_pendulum(t, state, l1=1.0, l2=1.0, m1=1.0, m2=1.0, g=9.81):

    theta1, omega1, theta2, omega2 = state

    # your code goes here:




    return 

In [ ]:
runtest(['test_double_pendulum'])

In [ ]:
from scipy.integrate import solve_ivp

# Initial condition: both rods pointing right at 90 degrees
theta0 = np.pi / 2
y0_dp  = [theta0, 0.0, theta0, 0.0]   # [theta1, omega1, theta2, omega2]

# Slightly perturbed initial condition
eps    = 1e-8
y0_dp2 = [theta0 + eps, 0.0, theta0, 0.0]

T_final = 30.0
t_eval  = np.linspace(0, T_final, 6001)

sol_1 = solve_ivp(double_pendulum, (0, T_final), y0_dp,
                  t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-11)
sol_2 = solve_ivp(double_pendulum, (0, T_final), y0_dp2,
                  t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-11)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))


# Panel 1: exponential divergence of theta1
ax = axes[0]
sep = np.abs(sol_1.y[0] - sol_2.y[0])
ax.semilogy(sol_1.t, sep + 1e-20, 'b-', lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('$|\\Delta\\theta_1|$')
ax.set_title(f'Divergence of two trajectories\n(initial sep = {eps:.0e} rad)', fontsize=11)
ax.grid(True, which='both', alpha=0.3)

# Panel 2: theta1(t) for both trajectories
ax = axes[1]
ax.plot(sol_1.t, sol_1.y[0], 'b-', lw=1.5, alpha=0.8, label='Original')
ax.plot(sol_2.t, sol_2.y[0], 'r-', lw=1.5, alpha=0.8, label='Perturbed by 0^{-8}$ rad')
ax.set_xlabel('Time (s)')
ax.set_ylabel('$\\theta_1$ (rad)')
ax.set_title('$\\theta_1(t)$: initially identical, then diverge', fontsize=11)
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Exercises

**1. Return map.** For a chaotic trajectory of the logistic map (e.g. $r = 3.9$), plot
$x_{n+1}$ vs $x_n$ for $n = 0, 1, \ldots, 1000$. What shape do you see? Overlay the
curve $x_{n+1} = rx_n(1-x_n)$. Make sure you plot individual points instead of a continuous line!

**2. Lyapunov spectrum.** Estimate the Lyapunov exponent of the logistic map for
$r \in [3, 4]$ and plot it against $r$. Where is $\lambda = 0$? How does this relate
to the bifurcation diagram?

**3. Lorenz attractor.** The Lorenz system $\dot{x} = \sigma(y-x),\,
\dot{y} = x(\rho - z) - y,\, \dot{z} = xy - \beta z$ (with $\sigma=10, \rho=28,
\beta=8/3$) is the canonical strange attractor. Integrate from $t=0$ to $t=50$
starting at $(1, 1, 1)$ and $(1+10^{-8}, 1, 1)$. Plot the 3D attractor and the
separation $|\delta r(t)|$.